In [ ]:
import os
os.environ["OPEN_API_KEY"]=""

In [ ]:
# Install Libraries

In [1]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
                faiss-cpu tiktoken python-dotenv

In [2]:
!pip install -U langchain faiss-cpu

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_3985/1760295084.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [13]:
# Indexing
video_id = "HWyzKmXquJk"

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id)

    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except Exception as e:
    print("Error:", e)

Heat. Heat. Heat. [Music] The morning light is turning blue. The feeling is bizarre. The night is almost over. I still don't know where you are. The shadows are they keep me pretty like a movie star. that makes me feel like you love [Music] in the end. I hope it's you and me in the darkness. I would never leave you in the light. [Music] The time has come. Now I'm educated and I do now [ __ ] my friends. Just get in the car. I just want to be right where you are. [Applause] [Music] Heat. Heat. [Music] Heat. Heat. [Music] Please, do you think about what it might be? Cuz I dream about you in my sleep. Heat. Heat. [Music] Heat. Heat. [Music] Heat. Heat. [Music] Heat. Heat. [Music] Heat. Heat. [Music]


In [14]:
transcript

"Heat. Heat. Heat. [Music] The morning light is turning blue. The feeling is bizarre. The night is almost over. I still don't know where you are. The shadows are they keep me pretty like a movie star. that makes me feel like you love [Music] in the end. I hope it's you and me in the darkness. I would never leave you in the light. [Music] The time has come. Now I'm educated and I do now [\xa0__\xa0] my friends. Just get in the car. I just want to be right where you are. [Applause] [Music] Heat. Heat. [Music] Heat. Heat. [Music] Please, do you think about what it might be? Cuz I dream about you in my sleep. Heat. Heat. [Music] Heat. Heat. [Music] Heat. Heat. [Music] Heat. Heat. [Music] Heat. Heat. [Music]"

In [15]:
# Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=70, chunk_overlap=14)
chunks = splitter.create_documents([transcript])

In [16]:
len(chunks)

13

In [17]:
chunks

[Document(metadata={}, page_content='Heat. Heat. Heat. [Music] The morning light is turning blue. The'),
 Document(metadata={}, page_content="blue. The feeling is bizarre. The night is almost over. I still don't"),
 Document(metadata={}, page_content="I still don't know where you are. The shadows are they keep me pretty"),
 Document(metadata={}, page_content='me pretty like a movie star. that makes me feel like you love [Music]'),
 Document(metadata={}, page_content="love [Music] in the end. I hope it's you and me in the darkness. I"),
 Document(metadata={}, page_content='darkness. I would never leave you in the light. [Music] The time has'),
 Document(metadata={}, page_content="The time has come. Now I'm educated and I do now [\xa0__\xa0] my friends."),
 Document(metadata={}, page_content='my friends. Just get in the car. I just want to be right where you'),
 Document(metadata={}, page_content='where you are. [Applause] [Music] Heat. Heat. [Music] Heat. Heat.'),
 Document(metadata={},

In [ ]:
# Embedding Generation & Storing in Vector Store

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_id(['']);

In [ ]:
# Retrival

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

In [ ]:
# Augmentation

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}

    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [ ]:
question = "is the topic of love song in this video? if yes then what are they telling"
retrieved_docs = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
# Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

In [ ]:
# Above is Done but need improvememts

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    "context" : retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [ ]:
# parallel_chain.invoke("is the topic of love song in this video? if yes then what are they telling")

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("is the topic of love song in this video? if yes then what are they telling")